##### Boilerplate

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/DumLLM"
NOTEBOOKS_DIR = os.path.join(PROJECT_DIR, "notebooks")
SRC_DIR = os.path.join(PROJECT_DIR, "src")

os.chdir(PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)
sys.path.insert(0, SRC_DIR)
sys.path.insert(0, NOTEBOOKS_DIR)

print("DumLLM initialized!")
print("Working directory:", os.getcwd())

Mounted at /content/drive
DumLLM initialized!
Working directory: /content/drive/MyDrive/DumLLM


In [3]:
import importlib
import sys

importlib.invalidate_caches()

import previous_chapters

importlib.reload(previous_chapters)

print([
    name for name in dir(previous_chapters)
    if not name.startswith("_")
])

['DataLoader', 'Dataset', 'FeedForward', 'GELU', 'GPTDatasetV1', 'GPTModel', 'LayerNorm', 'MultiHeadAttention', 'TransformerBlock', 'create_dataloader_v1', 'generate_text_simple', 'nn', 'tiktoken', 'torch']


# 5 Pretraning on unlabeled data

## 5.1 Evaluating generative text models

### 5.1.1 Using GPT to genenrate text

In [4]:
# import libraries
from importlib.metadata import version

pkgs = [
    "matplotlib",
    "numpy",
    "tiktoken",
    "torch",
    "tensorflow"          # for openAI's pretrained weights
]

for p in pkgs:
  print(f"{p} version: {version(p)}")

matplotlib version: 3.10.0
numpy version: 2.4.6
tiktoken version: 0.14.0
torch version: 2.11.0+cpu
tensorflow version: 2.21.0


In [11]:
from previous_chapters import GPTModel
import torch

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();

In [9]:
import tiktoken
from previous_chapters import generate_text_simple

def text_to_token_ids(text, tokenizer):
  encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
  encoded_tensor = torch.tensor(encoded).unsqueeze(0)   # add batch dimension
  return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
  flat = token_ids.squeeze(0)  # remove batch dimension
  return tokenizer.decode(flat.tolist())

In [10]:
start_context = "Every effort moves you"
tokenizer = tiktoken.encoding_for_model("gpt-2")
encoded_tensor = text_to_token_ids(start_context, tokenizer)
print(encoded_tensor)

token_ids_to_text(encoded_tensor, tokenizer)

tensor([[6109, 3626, 6100,  345]])


'Every effort moves you'

In [14]:
token_ids = generate_text_simple(
    model = model,
    idx = text_to_token_ids(start_context, tokenizer),
    max_new_tokens = 10,
    context_size = GPT_CONFIG_124M["context_length"]
)

In [16]:
token_ids.squeeze(0).shape

torch.Size([14])

In [17]:
token_ids_to_text(token_ids, tokenizer)

'Every effort moves you Aeiman Byeswickattributeometer inspector Normandy freezerigrate'

### 5.1.2 Calculating the text generation loss: cross-entropy and perplexity

In [18]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

In [19]:
with torch.no_grad():
  logits = model(inputs)

In [20]:
logits.shape

torch.Size([2, 3, 50257])

In [21]:
probas = torch.softmax(logits, dim=-1)
probas.shape

torch.Size([2, 3, 50257])

In [22]:
probas

tensor([[[1.5619e-05, 9.6721e-06, 2.3829e-05,  ..., 1.9601e-05,
          2.7812e-05, 5.2847e-06],
         [2.1799e-05, 7.4676e-06, 8.7542e-06,  ..., 1.1746e-05,
          2.4788e-05, 1.4412e-05],
         [4.0975e-05, 1.2116e-05, 1.9115e-05,  ..., 1.8165e-05,
          1.2299e-05, 1.4943e-05]],

        [[2.8877e-05, 1.2905e-05, 4.2591e-05,  ..., 9.8521e-06,
          3.4002e-05, 7.1068e-06],
         [2.0209e-05, 2.4667e-05, 1.9506e-05,  ..., 6.1039e-06,
          3.2126e-05, 1.3260e-05],
         [3.1132e-05, 1.9669e-05, 2.4862e-05,  ..., 1.2855e-05,
          1.4016e-05, 2.1325e-05]]])

In [23]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("Token IDs:\n", token_ids)

Token IDs:
 tensor([[[36397],
         [39619],
         [20610]],

        [[ 8615],
         [49289],
         [47105]]])


In [25]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

Targets batch 1:  effort moves you
Outputs batch 1:  Gathering SerbianFriday


In [26]:
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 1:",target_probas_1)

Text 1: tensor([2.3466e-05, 2.0531e-05, 1.1733e-05])


In [27]:
text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 2:",target_probas_2)

Text 2: tensor([4.2794e-05, 1.6248e-05, 1.1586e-05])


In [31]:
# compute the log of all token probabilities
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)

tensor([-10.6600, -10.7936, -11.3531, -10.0591, -11.0276, -11.3658])


In [33]:
-1 * torch.mean(log_probas)

tensor(10.8765)

In [36]:
logits_flat = logits.flatten(0, 1)
logits_flat.shape

torch.Size([6, 50257])

In [37]:
targets_flat = targets.flatten(0, 1)
targets_flat.shape

torch.Size([6])

In [38]:
torch.nn.functional.cross_entropy(logits_flat, targets_flat)

tensor(10.8765)